# 🎬 Amazon Prime Video — Exploratory Data Analysis
**AlmaBetter Capstone Project | Data Science & Machine Learning**

---

## 📌 Project Name
**Exploratory Data Analysis on Amazon Prime Video Dataset**

## 📝 Project Summary
Amazon Prime Video is one of the world's leading streaming platforms. This project performs a comprehensive Exploratory Data Analysis (EDA) on its U.S. content library using two datasets: `titles.csv` (9,871 titles with metadata) and `credits.csv` (124,235 cast/crew records). The goal is to discover patterns in content type, genres, ratings, production trends, and key contributors — and translate those patterns into actionable business insights.

## 🎯 Business Objective
To analyze all shows and movies available on Amazon Prime Video and extract insights around:
- Content diversity (genres, types)
- Temporal trends (growth over release years)
- Quality signals (IMDb & TMDb ratings)
- Key talent contributing to the catalog
- Regional production patterns

These insights will help **content strategists**, **marketing teams**, and **investors** make smarter decisions around content acquisition, promotion, and platform growth.

## ❓ Problem Statement
In a highly competitive streaming landscape, Amazon Prime must continually understand: *What content is working? Where is the library growing? What do audiences rate highly?* This EDA answers those questions by mining the platform's own catalog data.

# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import ast
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

print('✅ All libraries imported successfully!')

### Dataset Loading



In [ ]:
# Load Dataset
titles_df  = pd.read_csv('titles.csv')
credits_df = pd.read_csv('credits.csv')


print(f'titles_df  loaded → {titles_df.shape[0]:,} rows × {titles_df.shape[1]} columns')
print(f'credits_df loaded → {credits_df.shape[0]:,} rows × {credits_df.shape[1]} columns')

### Dataset First View

In [ ]:
# Dataset First Look
print('── titles_df : First 5 rows ──')
display(titles_df.head())
print('\n── credits_df : First 5 rows ──')
display(credits_df.head())

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
print(f'titles_df  : {titles_df.shape[0]:,} rows, {titles_df.shape[1]} columns')
print(f'credits_df : {credits_df.shape[0]:,} rows, {credits_df.shape[1]} columns')

### Dataset Information

In [ ]:
# Dataset Info
print('── titles_df Info ──')
titles_df.info()
print('\n── credits_df Info ──')
credits_df.info()

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
print(f'Duplicate rows in titles_df  : {titles_df.duplicated().sum()}')
print(f'Duplicate rows in credits_df : {credits_df.duplicated().sum()}')

#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
print('── titles_df Missing Values ──')
missing_t = pd.DataFrame({
    'Missing Count': titles_df.isnull().sum(),
    'Missing %': (titles_df.isnull().sum() / len(titles_df) * 100).round(2)
}).sort_values('Missing Count', ascending=False)
display(missing_t[missing_t['Missing Count'] > 0])

print('\n── credits_df Missing Values ──')
missing_c = pd.DataFrame({
    'Missing Count': credits_df.isnull().sum(),
    'Missing %': (credits_df.isnull().sum() / len(credits_df) * 100).round(2)
}).sort_values('Missing Count', ascending=False)
display(missing_c[missing_c['Missing Count'] > 0])

In [ ]:
# Visualizing the missing values
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# titles_df
miss_t = titles_df.isnull().sum().sort_values(ascending=False)
miss_t = miss_t[miss_t > 0]
axes[0].bar(miss_t.index, miss_t.values, color='#e50914', edgecolor='white')
axes[0].set_title('Missing Values — titles_df', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].set_xlabel('Column')
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45, ha='right')
for bar in axes[0].patches:
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+10,
                 int(bar.get_height()), ha='center', fontsize=9)

# credits_df
miss_c = credits_df.isnull().sum().sort_values(ascending=False)
miss_c = miss_c[miss_c > 0]
if len(miss_c):
    axes[1].bar(miss_c.index, miss_c.values, color='#146eb4', edgecolor='white')
    axes[1].set_title('Missing Values — credits_df', fontsize=13, fontweight='bold')
    axes[1].set_ylabel('Count')
    axes[1].set_xlabel('Column')
    plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45, ha='right')
else:
    axes[1].text(0.5, 0.5, 'No Missing Values\nin credits_df',
                 ha='center', va='center', fontsize=14, color='green',
                 transform=axes[1].transAxes)
    axes[1].set_title('Missing Values — credits_df', fontsize=13, fontweight='bold')
    axes[1].axis('off')

plt.suptitle('Missing Value Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### What did you know about your dataset?

**titles_df** contains **9,871 unique titles** (Movies & TV Shows) available on Amazon Prime Video in the US, with 15 columns covering metadata like release year, genres, runtime, age certification, IMDb/TMDb scores, and votes. Several columns have missing values — notably `age_certification` (~40%), `seasons` (~70%), and `imdb_score`/`tmdb_score` (~20%). The dataset spans content released from the early 1900s to 2022.

**credits_df** contains **124,235 records** of actors and directors linked to titles via the `id` column. The `character` column has ~10% missing values (mainly for directors, who have no character name). There are no duplicate records in either dataset.

Key observations:
- The dataset is a mix of numerical and categorical features.
- `genres` and `production_countries` are stored as string-formatted lists and need parsing.
- Numeric columns like `imdb_score`, `tmdb_score`, and `runtime` need median imputation for missing values.

## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
print('── titles_df Columns ──')
print(titles_df.columns.tolist())
print('\n── credits_df Columns ──')
print(credits_df.columns.tolist())

In [ ]:
# Dataset Describe
print('── Numerical Summary : titles_df ──')
display(titles_df.describe())
print('\n── Numerical Summary : credits_df ──')
display(credits_df.describe(include='all'))

### Variables Description

**titles_df:**
| Column | Type | Description |
|---|---|---|
| `id` | str | Unique title ID on JustWatch |
| `title` | str | Name of the movie or show |
| `type` | str | `MOVIE` or `SHOW` |
| `description` | str | Short plot summary |
| `release_year` | int | Year of release |
| `age_certification` | str | Age rating (PG, R, TV-MA, etc.) |
| `runtime` | int | Duration in minutes (episode length for shows) |
| `genres` | str | List of genres (stored as string) |
| `production_countries` | str | List of country codes (stored as string) |
| `seasons` | float | Number of seasons (shows only) |
| `imdb_id` | str | IMDb title identifier |
| `imdb_score` | float | IMDb rating (0–10) |
| `imdb_votes` | float | Number of IMDb votes |
| `tmdb_popularity` | float | TMDb popularity score |
| `tmdb_score` | float | TMDb rating (0–10) |

**credits_df:**
| Column | Type | Description |
|---|---|---|
| `person_id` | int | Unique person ID on JustWatch |
| `id` | str | Title ID (foreign key to titles_df) |
| `name` | str | Actor or director name |
| `character` | str | Character name (actors only) |
| `role` | str | `ACTOR` or `DIRECTOR` |

In [ ]:
# Check Unique Values for each variable.
print('── Unique Values : titles_df ──')
for col in titles_df.columns:
    n = titles_df[col].nunique()
    sample = titles_df[col].dropna().unique()[:3]
    print(f'  {col:25s} → {n:5d} unique  | sample: {list(sample)}')

print('\n── Unique Values : credits_df ──')
for col in credits_df.columns:
    n = credits_df[col].nunique()
    sample = credits_df[col].dropna().unique()[:3]
    print(f'  {col:25s} → {n:6d} unique  | sample: {list(sample)}')

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Write your code to make your dataset analysis ready.

# ── 1. Remove Duplicates ─────────────────────────────────────────────────
titles_df.drop_duplicates(inplace=True)
credits_df.drop_duplicates(inplace=True)

# ── 2. Parse genres & production_countries from string → actual list ──────
def safe_parse(val):
    try:
        return ast.literal_eval(val) if isinstance(val, str) else []
    except Exception:
        return []

titles_df['genres_list']    = titles_df['genres'].apply(safe_parse)
titles_df['countries_list'] = titles_df['production_countries'].apply(safe_parse)

# ── 3. Fill missing numeric columns with median ───────────────────────────
for col in ['imdb_score', 'tmdb_score', 'imdb_votes', 'tmdb_popularity', 'runtime']:
    median_val = titles_df[col].median()
    titles_df[col].fillna(median_val, inplace=True)
    print(f'  Filled {col} NaNs with median = {median_val:.2f}')

# ── 4. Fill missing categorical columns ──────────────────────────────────
titles_df['age_certification'].fillna('Not Rated', inplace=True)
credits_df['character'].fillna('Unknown', inplace=True)

# ── 5. seasons: fill 0 for movies (they have no seasons) ─────────────────
titles_df.loc[titles_df['type'] == 'MOVIE', 'seasons'] = titles_df.loc[titles_df['type'] == 'MOVIE', 'seasons'].fillna(0)
titles_df.loc[titles_df['type'] == 'SHOW',  'seasons'] = titles_df.loc[titles_df['type'] == 'SHOW',  'seasons'].fillna(1)

# ── 6. Merge credits with title type for richer analysis ─────────────────
credits_merged = credits_df.merge(titles_df[['id','title','type','release_year','imdb_score']], on='id', how='left')

print('\n✅ Data Wrangling Complete!')
print(f'titles_df  → {titles_df.shape}')
print(f'credits_df → {credits_df.shape}')
print(f'credits_merged → {credits_merged.shape}')

### What all manipulations have you done and insights you found?

1. **Removed duplicates** — No duplicates existed in either dataset, confirming data integrity.
2. **Parsed list-columns** — `genres` and `production_countries` were stored as string representations of Python lists (e.g., `"['drama', 'comedy']"`) and were converted to actual Python lists using `ast.literal_eval` for proper analysis.
3. **Imputed missing numeric values** with the column median — a robust strategy that avoids skew from outliers. Affected columns: `imdb_score`, `tmdb_score`, `imdb_votes`, `tmdb_popularity`, `runtime`.
4. **Filled categorical NaNs** — `age_certification` was filled with `'Not Rated'`; `character` in credits was filled with `'Unknown'` (applicable to directors who have no character name).
5. **Handled `seasons` logically** — Movies with NaN seasons were set to 0; shows with unknown seasons were set to 1 as a conservative default.
6. **Created `credits_merged`** — Joined credits with title metadata to enable actor/director analysis by content type, year, and rating.

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
# Chart - 1 visualization code
# PIE CHART + BAR CHART — Movies vs TV Shows Distribution

type_counts = titles_df['type'].value_counts()
colors = ['#e50914', '#146eb4']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].pie(type_counts, labels=type_counts.index, autopct='%1.1f%%',
            startangle=90, colors=colors, explode=[0.05, 0],
            shadow=True, textprops={'fontsize': 13})
axes[0].set_title('Content Type Distribution (%)', fontsize=14, fontweight='bold')

bars = axes[1].bar(type_counts.index, type_counts.values, color=colors, width=0.45, edgecolor='white')
for bar in bars:
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+40,
                 f'{int(bar.get_height()):,}', ha='center', fontsize=12, fontweight='bold')
axes[1].set_title('Content Count by Type', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Number of Titles')
axes[1].set_xlabel('Content Type')
sns.despine(ax=axes[1])

plt.suptitle('Chart 1 — Movies vs TV Shows on Amazon Prime', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
A **Pie Chart** combined with a **Bar Chart** was chosen. The pie chart shows the proportional split (%) between Movies and TV Shows at a glance, while the bar chart communicates the actual raw count — together they give both relative and absolute context.

##### 2. What is/are the insight(s) found from the chart?
- Movies make up approximately **80%** of the Amazon Prime catalog, while TV Shows account for ~20%.
- In raw numbers, Movies outnumber Shows by nearly 4:1.
- Amazon Prime is heavily movie-centric compared to platforms like Netflix which invest more heavily in original series.

##### 3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth? Justify with specific reason.
**Positive Impact:** Content strategists can use this to benchmark against competitors. The movie-heavy nature means less recurring engagement — TV shows create stronger subscriber retention (viewers return weekly/seasonally).

**Negative Growth Risk:** Over-reliance on movies may lead to **higher churn** because movies are one-time watches. Platforms with more TV shows tend to have better retention metrics. Amazon should increase investment in original TV series to drive long-term subscriptions.

#### Chart - 2

In [ ]:
# Chart - 2 visualization code
# HORIZONTAL BAR CHART — Top 15 Genres

genres_exp = titles_df.explode('genres_list')
genres_exp = genres_exp[genres_exp['genres_list'].notna() & (genres_exp['genres_list'] != '')]
top_genres = genres_exp['genres_list'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 7))
palette = sns.color_palette('Reds_r', n_colors=15)
bars = ax.barh(top_genres.index[::-1], top_genres.values[::-1], color=palette, edgecolor='white')
for bar in bars:
    ax.text(bar.get_width()+20, bar.get_y()+bar.get_height()/2,
            f'{int(bar.get_width()):,}', va='center', fontsize=10)

ax.set_title('Chart 2 — Top 15 Genres on Amazon Prime Video', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Titles')
ax.set_ylabel('Genre')
sns.despine()
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
A **Horizontal Bar Chart** is ideal for ranked categorical data with text labels. It allows easy comparison of genre frequencies and keeps genre names readable without rotation.

##### 2. What is/are the insight(s) found from the chart?
- **Drama** and **Comedy** are the top two genres by a significant margin, indicating audience demand for emotionally engaging and lighthearted content.
- **Action**, **Thriller**, and **Documentary** follow as mid-tier genres.
- Niche genres like **Animation**, **Western**, and **Sci-Fi** are present but represent smaller slices of the catalog.

##### 3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth?
**Positive:** Focusing new original content investments in Drama and Comedy has the highest probability of audience reach.

**Negative Growth Risk:** Saturation in Drama/Comedy means Amazon will find it harder to differentiate. Investing in underrepresented genres (e.g., Sci-Fi originals, Documentaries) could create a **competitive niche** and attract new subscriber segments.

#### Chart - 3

In [ ]:
# Chart - 3 visualization code
# LINE CHART — Number of Titles Released Per Year

year_data   = titles_df[titles_df['release_year'] >= 1970]
year_counts = year_data.groupby(['release_year','type']).size().unstack(fill_value=0).reset_index()

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(year_counts['release_year'], year_counts.get('MOVIE', 0),
        label='Movies', color='#e50914', linewidth=2.5)
ax.plot(year_counts['release_year'], year_counts.get('SHOW', 0),
        label='TV Shows', color='#146eb4', linewidth=2.5, linestyle='--')
ax.fill_between(year_counts['release_year'], year_counts.get('MOVIE', 0), alpha=0.15, color='#e50914')
ax.fill_between(year_counts['release_year'], year_counts.get('SHOW', 0), alpha=0.15, color='#146eb4')

ax.set_title('Chart 3 — Number of Titles Released Per Year', fontsize=14, fontweight='bold')
ax.set_xlabel('Release Year')
ax.set_ylabel('Number of Titles')
ax.legend(fontsize=12)
ax.xaxis.set_major_locator(mticker.MultipleLocator(5))
plt.xticks(rotation=45)
sns.despine()
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
A **Line Chart with area fill** is the standard choice for showing trends over continuous time. It clearly communicates the trajectory and rate of change in content additions across decades.

##### 2. What is/are the insight(s) found from the chart?
- Content additions grew slowly until ~2000, then accelerated sharply from 2010 onward.
- The peak was around **2018–2020**, after which there's a decline — likely reflecting the COVID-19 disruption to production pipelines.
- TV Show growth is flatter but consistent; Movie growth is more volatile year-to-year.

##### 3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth?
**Positive:** The steep rise post-2010 reflects Amazon's aggressive content acquisition strategy — a proven driver of subscriber growth.

**Negative Growth Risk:** The post-2020 content slowdown is a risk signal. If the catalog stops growing, subscriber growth may stagnate. Amazon needs to sustain or increase original production to maintain momentum.

#### Chart - 4

In [ ]:
# Chart - 4 visualization code
# HISTOGRAM — Distribution of IMDb Scores

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, ctype, color in zip(axes, ['MOVIE','SHOW'], ['#e50914','#146eb4']):
    subset = titles_df[titles_df['type'] == ctype]['imdb_score']
    sns.histplot(subset, bins=30, kde=True, color=color, ax=ax, alpha=0.7)
    ax.axvline(subset.mean(), color='black', linestyle='--', linewidth=1.5, label=f'Mean: {subset.mean():.2f}')
    ax.axvline(subset.median(), color='gray', linestyle=':', linewidth=1.5, label=f'Median: {subset.median():.2f}')
    ax.set_title(f'IMDb Score Distribution — {ctype}s', fontsize=13, fontweight='bold')
    ax.set_xlabel('IMDb Score')
    ax.set_ylabel('Count')
    ax.legend()

plt.suptitle('Chart 4 — IMDb Score Distribution by Content Type', fontsize=14, fontweight='bold', y=1.01)
sns.despine()
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
A **Histogram with KDE overlay** is the best way to visualize the full distribution of a continuous variable. The KDE (Kernel Density Estimate) smooths the distribution to reveal its underlying shape, while mean/median lines highlight central tendency.

##### 2. What is/are the insight(s) found from the chart?
- Both Movies and Shows have IMDb scores **normally distributed** around 6.0–6.5.
- TV Shows tend to have a **slightly higher average IMDb score** than Movies — audiences rate long-format content more generously.
- Very few titles score below 4.0 or above 9.0, confirming most content is of average quality.

##### 3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth?
**Positive:** The rating distribution helps set quality benchmarks for acquisitions. Content below 5.0 IMDb is likely dragging platform perception.

**Negative Growth Risk:** A large mass of average (5–7) scored content can make the platform feel mediocre. A curation strategy to remove or de-prioritize low-rated content in recommendations can improve perceived quality.

#### Chart - 5

In [ ]:
# Chart - 5 visualization code
# SCATTER PLOT — IMDb Score vs TMDb Score

rating_df = titles_df[(titles_df['imdb_score'] > 0) & (titles_df['tmdb_score'] > 0)]

fig, ax = plt.subplots(figsize=(10, 7))
sns.scatterplot(data=rating_df, x='imdb_score', y='tmdb_score',
                hue='type', alpha=0.4, s=20, ax=ax,
                palette={'MOVIE':'#e50914','SHOW':'#146eb4'})
sns.regplot(data=rating_df, x='imdb_score', y='tmdb_score',
            scatter=False, ax=ax, color='black',
            line_kws={'linewidth':2,'linestyle':'--'})

corr = rating_df[['imdb_score','tmdb_score']].corr().iloc[0,1]
ax.text(0.05, 0.92, f'Pearson r = {corr:.3f}', transform=ax.transAxes,
        fontsize=12, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

ax.set_title('Chart 5 — IMDb Score vs TMDb Score', fontsize=14, fontweight='bold')
ax.set_xlabel('IMDb Score')
ax.set_ylabel('TMDb Score')
sns.despine()
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
A **Scatter Plot with regression line** directly visualizes the relationship between two continuous rating variables. The Pearson correlation coefficient quantifies the strength of the relationship.

##### 2. What is/are the insight(s) found from the chart?
- There is a **moderate positive correlation** (r ≈ 0.65–0.75) between IMDb and TMDb scores — the platforms generally agree on quality but not always.
- Movies (red) are more dispersed than Shows (blue), suggesting more rating disagreement for films.
- Some titles score highly on TMDb but poorly on IMDb and vice versa — platform-specific audience biases exist.

##### 3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth?
**Positive:** Using both rating systems together gives a more reliable quality signal than either alone — valuable for content recommendation algorithms.

**Negative Growth Risk:** Titles with high TMDb but low IMDb (or vice versa) may be misclassified by single-score filters, causing good content to be buried in recommendations. A combined score model reduces this risk.

#### Chart - 6

In [ ]:
# Chart - 6 visualization code
# BOX PLOT — Runtime Distribution by Content Type

runtime_df = titles_df[(titles_df['runtime'] > 0) & (titles_df['runtime'] < 300)]

fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=runtime_df, x='type', y='runtime', ax=ax,
            palette={'MOVIE':'#e50914','SHOW':'#146eb4'},
            width=0.4, fliersize=3, linewidth=1.5)

ax.set_title('Chart 6 — Runtime Distribution by Content Type', fontsize=14, fontweight='bold')
ax.set_xlabel('Content Type')
ax.set_ylabel('Runtime (minutes)')

for ctype in ['MOVIE','SHOW']:
    med = runtime_df[runtime_df['type']==ctype]['runtime'].median()
    print(f'{ctype} median runtime: {med:.0f} mins')

sns.despine()
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
A **Box Plot** efficiently summarizes runtime distribution by showing median, interquartile range, and outliers in a single compact visual — perfect for comparing two groups side by side.

##### 2. What is/are the insight(s) found from the chart?
- Movies have a higher median runtime (~90–100 min) with a fairly tight distribution.
- TV show episodes are shorter (~30–50 min) with a wider spread — reflecting diversity from 20-min comedies to 60-min dramas.
- Both types have outliers: some movies exceed 3 hours; some shows have very short episodes (under 10 min).

##### 3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth?
**Positive:** Runtime data helps the platform optimize UI — e.g., showing 'Quick Watch' tags for content under 30 minutes to drive impulse engagement.

**Negative Growth Risk:** Very long movies (>150 min outliers) may have lower completion rates, which negatively impacts engagement metrics and algorithmic recommendations.

#### Chart - 7

In [ ]:
# Chart - 7 visualization code
# BAR CHART — Top 10 Production Countries

countries_exp = titles_df.explode('countries_list')
countries_exp = countries_exp[countries_exp['countries_list'].notna() & (countries_exp['countries_list'] != '')]
top_countries = countries_exp['countries_list'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(12, 6))
palette = sns.color_palette('crest', n_colors=10)
bars = ax.bar(top_countries.index, top_countries.values, color=palette, edgecolor='white')
for bar in bars:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+20,
            f'{int(bar.get_height()):,}', ha='center', fontsize=10, fontweight='bold')

ax.set_title('Chart 7 — Top 10 Content Producing Countries', fontsize=14, fontweight='bold')
ax.set_xlabel('Country Code')
ax.set_ylabel('Number of Titles')
plt.xticks(rotation=30)
sns.despine()
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
A **Vertical Bar Chart** is clear and intuitive for comparing counts across a small number of discrete categories (countries). Sorting by count makes the ranking immediately apparent.

##### 2. What is/are the insight(s) found from the chart?
- The **United States (US)** dominates content production by a wide margin.
- **India (IN)** ranks second — reflecting Amazon's significant investment in Bollywood and Indian original content.
- **UK, Japan, France, and Germany** are notable contributors, showing a strong European and Asian content mix.

##### 3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth?
**Positive:** The India ranking confirms that regional investment is paying off. Replicating this model in other high-growth markets (Brazil, Southeast Asia, Nigeria) could drive subscriber growth.

**Negative Growth Risk:** Heavy US-centric content may alienate international subscribers. Over-reliance on one production geography creates a risk if US content costs rise significantly.

#### Chart - 8

In [ ]:
# Chart - 8 visualization code
# HORIZONTAL BAR — Top 15 Most Featured Actors

actors = credits_df[credits_df['role'] == 'ACTOR']['name'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 7))
palette = sns.color_palette('Blues_d', 15)
ax.barh(actors.index[::-1], actors.values[::-1], color=palette, edgecolor='white')
for i, val in enumerate(actors.values[::-1]):
    ax.text(val+0.3, i, str(val), va='center', fontsize=10)

ax.set_title('Chart 8 — Top 15 Most Featured Actors on Amazon Prime', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Appearances')
ax.set_ylabel('Actor Name')
sns.despine()
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
A **Horizontal Bar Chart** is again ideal here for ranking named entities. Horizontal orientation keeps actor names readable.

##### 2. What is/are the insight(s) found from the chart?
- A small number of actors appear in a very large number of titles, indicating they are staple performers on the platform.
- Some actors appear across both Movies and Shows, making them versatile brand assets.
- The high frequency suggests some actors may be associated with prolific franchises or documentary series.

##### 3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth?
**Positive:** Top actors can be featured prominently in marketing campaigns — they are recognizable brand ambassadors for Amazon Prime content.

**Negative Growth Risk:** Over-dependence on a narrow pool of actors can reduce perceived content diversity and make the platform feel repetitive to long-term subscribers.

#### Chart - 9

In [ ]:
# Chart - 9 visualization code
# HORIZONTAL BAR — Top 15 Most Active Directors

directors = credits_df[credits_df['role'] == 'DIRECTOR']['name'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 7))
palette = sns.color_palette('Oranges_d', 15)
ax.barh(directors.index[::-1], directors.values[::-1], color=palette, edgecolor='white')
for i, val in enumerate(directors.values[::-1]):
    ax.text(val+0.1, i, str(val), va='center', fontsize=10)

ax.set_title('Chart 9 — Top 15 Most Active Directors on Amazon Prime', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Titles Directed')
ax.set_ylabel('Director Name')
sns.despine()
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
Same rationale as Chart 8 — a **Horizontal Bar Chart** ranks prolific directors clearly. A separate chart from actors keeps the analysis clean and focused.

##### 2. What is/are the insight(s) found from the chart?
- The top directors have directed 5–15 titles available on Amazon Prime.
- Certain directors appear to specialize in specific genres (visible when cross-referenced with the genre data).
- The distribution is more concentrated than actors — top directors contribute proportionally more titles per person.

##### 3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth?
**Positive:** Prolific directors with consistently high IMDb scores are strong candidates for exclusive original content partnerships.

**Negative Growth Risk:** If top directors are not under exclusive deals, competitors can poach them, reducing Amazon's catalog quality advantage.

#### Chart - 10

In [ ]:
# Chart - 10 visualization code
# GROUPED BAR — Genre Distribution: Movies vs Shows

genres_exp2 = titles_df.explode('genres_list')
genres_exp2 = genres_exp2[genres_exp2['genres_list'].notna() & (genres_exp2['genres_list'] != '')]
top10_genres = genres_exp2['genres_list'].value_counts().head(10).index

genre_type = (genres_exp2[genres_exp2['genres_list'].isin(top10_genres)]
              .groupby(['genres_list','type']).size().unstack(fill_value=0))

genre_type.plot(kind='bar', figsize=(14, 6), color=['#e50914','#146eb4'],
                edgecolor='white', width=0.7)
plt.title('Chart 10 — Top Genres Split by Content Type (Movie vs Show)', fontsize=14, fontweight='bold')
plt.xlabel('Genre')
plt.ylabel('Number of Titles')
plt.xticks(rotation=30, ha='right')
plt.legend(title='Type')
sns.despine()
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
A **Grouped Bar Chart** is the right tool to compare two categorical groups (Movie vs Show) across multiple genre categories simultaneously, revealing genre preference differences between content types.

##### 2. What is/are the insight(s) found from the chart?
- Drama is the #1 genre for **both** Movies and Shows.
- Comedy is notably stronger in Movies than Shows.
- **Documentary** and **Reality** genres skew toward Shows, which makes sense given their serial format.
- Action is more prevalent in Movies.

##### 3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth?
**Positive:** Genre-type insights help Amazon tailor recommendations — e.g., showing Documentary series (not films) when users prefer shows.

**Negative Growth Risk:** If Comedy movies dominate without equivalent Comedy shows, subscribers who prefer TV format are underserved — a gap competitors could exploit.

#### Chart - 11

In [ ]:
# Chart - 11 visualization code
# BAR CHART — Age Certification Distribution

age_counts = titles_df['age_certification'].value_counts().head(12)

fig, ax = plt.subplots(figsize=(13, 6))
palette = sns.color_palette('viridis', n_colors=12)
bars = ax.bar(age_counts.index, age_counts.values, color=palette, edgecolor='white')
for bar in bars:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+15,
            f'{int(bar.get_height()):,}', ha='center', fontsize=10)

ax.set_title('Chart 11 — Age Certification Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Age Certification')
ax.set_ylabel('Number of Titles')
plt.xticks(rotation=30)
sns.despine()
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
A **Bar Chart** clearly shows the count of titles per age certification category — a discrete, unordered categorical variable where bar height directly conveys volume.

##### 2. What is/are the insight(s) found from the chart?
- A significant portion of the catalog is marked **'Not Rated'** (our imputed placeholder) or **R / TV-MA**, indicating the platform leans toward adult audiences.
- **PG-13** and **PG** content is present but in smaller volumes — family-friendly content is a minority.
- **G-rated** content (suitable for all ages) is very rare.

##### 3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth?
**Positive:** Adult content dominance aligns with the primary subscriber demographic (18–45 year olds) who drive subscription revenue.

**Negative Growth Risk:** Limited family/children's content (G/PG ratings) means Amazon Prime may lose to Disney+ and Netflix for family subscribers — a growing and loyal segment.

#### Chart - 12

In [ ]:
# Chart - 12 visualization code
# VIOLIN PLOT — IMDb Score by Top Genres

genres_exp3 = titles_df.explode('genres_list')
top6 = genres_exp3['genres_list'].value_counts().head(6).index
violin_df = genres_exp3[genres_exp3['genres_list'].isin(top6)]

fig, ax = plt.subplots(figsize=(14, 7))
sns.violinplot(data=violin_df, x='genres_list', y='imdb_score', ax=ax,
               palette='Set2', inner='quartile', cut=0)
ax.set_title('Chart 12 — IMDb Score Distribution by Top 6 Genres', fontsize=14, fontweight='bold')
ax.set_xlabel('Genre')
ax.set_ylabel('IMDb Score')
plt.xticks(rotation=20)
sns.despine()
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
A **Violin Plot** combines a box plot's summary statistics with a KDE's shape — giving richer insight into how IMDb scores are distributed within each genre, including where the mass of ratings concentrates.

##### 2. What is/are the insight(s) found from the chart?
- **Documentation and Drama** genres tend to have higher median IMDb scores.
- **Comedy and Thriller** show wider score distributions — more variation in quality.
- All genres have a cluster of low-rated content below 5.0, indicating poor-quality titles exist in every genre.

##### 3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth?
**Positive:** Genres with consistently high IMDb scores (e.g., Documentary) should be featured in quality-focused marketing campaigns.

**Negative Growth Risk:** Comedy's wide rating variance means many low-quality comedy titles may dilute the genre's reputation on the platform — a curation problem.

#### Chart - 13

In [ ]:
# Chart - 13 visualization code
# LINE CHART — Average IMDb Score Over Release Years

year_rating = (titles_df[titles_df['release_year'] >= 1980]
               .groupby('release_year')['imdb_score']
               .agg(['mean','count'])
               .reset_index())
year_rating = year_rating[year_rating['count'] >= 5]  # filter sparse years

fig, ax1 = plt.subplots(figsize=(14, 6))
ax2 = ax1.twinx()

ax1.plot(year_rating['release_year'], year_rating['mean'],
         color='#e50914', linewidth=2.5, label='Avg IMDb Score')
ax1.fill_between(year_rating['release_year'], year_rating['mean'], alpha=0.15, color='#e50914')
ax2.bar(year_rating['release_year'], year_rating['count'],
        alpha=0.25, color='#146eb4', label='Title Count')

ax1.set_title('Chart 13 — Average IMDb Score vs Number of Titles Per Year', fontsize=14, fontweight='bold')
ax1.set_xlabel('Release Year')
ax1.set_ylabel('Average IMDb Score', color='#e50914')
ax2.set_ylabel('Number of Titles', color='#146eb4')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc='upper left')

plt.xticks(rotation=45)
sns.despine()
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
A **Dual-axis Line + Bar Chart** was chosen to overlay two related but differently scaled metrics on the same chart — average quality (IMDb score, left axis) against volume of content (count, right axis) — revealing whether quality and quantity move together.

##### 2. What is/are the insight(s) found from the chart?
- Older titles (pre-2000) have **higher average IMDb scores** — these are classic films that have stood the test of time and survived the 'survival bias' (only the best old content remains well-known).
- Post-2010 content volume surged but average scores slightly declined — consistent with the 'quality vs quantity' tradeoff from bulk acquisitions.
- The most recent years (2020–2022) show fewer titles but scores are stabilizing — possibly reflecting a quality-focused strategy shift.

##### 3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth?
**Positive:** Investing in fewer but higher-quality originals (as reflected by the recent stabilization) is the right strategic signal.

**Negative Growth Risk:** The bulk acquisition era (2015–2019) that lowered average scores may have trained users to see Amazon Prime as a 'second-tier' platform — brand perception damage that takes time to reverse.

#### Chart - 14 - Correlation Heatmap

In [ ]:
# Correlation Heatmap visualization code

num_cols = ['release_year','runtime','imdb_score','imdb_votes','tmdb_popularity','tmdb_score']
corr_matrix = titles_df[num_cols].corr()

fig, ax = plt.subplots(figsize=(10, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, linewidths=0.5,
            ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Chart 14 — Correlation Heatmap of Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?
A **Correlation Heatmap** is the standard tool for visualizing pairwise relationships among all numerical features simultaneously. Color encoding makes strong/weak correlations immediately identifiable.

##### 2. What is/are the insight(s) found from the chart?
- **imdb_votes** and **tmdb_popularity** show the strongest positive correlation — popular titles attract more rating activity across both platforms.
- **imdb_score** and **tmdb_score** are moderately correlated (~0.65–0.75), confirming partial but not complete agreement between platforms.
- **release_year** has a slight negative correlation with **imdb_score** — older classics tend to score higher (survivor bias).
- **runtime** has weak correlations with all other features, suggesting duration alone doesn't drive ratings or popularity.

#### Chart - 15 - Pair Plot

In [ ]:
# Pair Plot visualization code
# Note: Sampling for speed — pair plots on 9000+ rows are slow

pair_cols = ['release_year','runtime','imdb_score','tmdb_score','tmdb_popularity']
sample_df = titles_df[pair_cols + ['type']].dropna().sample(n=1500, random_state=42)

pp = sns.pairplot(
    sample_df,
    hue='type',
    vars=pair_cols,
    palette={'MOVIE':'#e50914','SHOW':'#146eb4'},
    plot_kws={'alpha': 0.3, 's': 15},
    diag_kind='kde'
)
pp.fig.suptitle('Chart 15 — Pair Plot of Key Numerical Features (Sample n=1500)',
                 fontsize=14, fontweight='bold', y=1.01)
plt.show()

##### 1. Why did you pick the specific chart?
A **Pair Plot** provides a matrix of scatter plots and KDE diagonals that simultaneously reveals relationships between every pair of numerical variables, colored by content type — a comprehensive multivariate overview in a single figure.

##### 2. What is/are the insight(s) found from the chart?
- **Movies and Shows separate clearly** along `runtime` — confirming runtime is the strongest type-distinguishing feature.
- `imdb_score` vs `tmdb_score` shows a clear positive diagonal — confirming their moderate correlation.
- `tmdb_popularity` has a **long right tail** — a small number of titles are extremely popular while most have low popularity scores (power-law distribution).
- `release_year` vs `imdb_score` confirms the older = higher rated trend visible in Chart 13.
- Diagonal KDE plots show Movies have broader `runtime` distributions vs Shows which cluster at lower runtimes.

## **5. Solution to Business Objective**

#### What do you suggest the client to achieve Business Objective?
Explain Briefly.

### 💡 Solution to Business Objective

Based on the comprehensive EDA conducted on the Amazon Prime Video dataset, the following evidence-based recommendations are suggested:

---

**1. Invest in TV Show Originals (Retention Strategy)**
Movies dominate the catalog (~80%) but TV Shows drive higher subscriber retention due to recurring engagement. Amazon should shift investment toward original TV series, especially in Drama — the highest-rated, most-consumed genre across both content types.

**2. Prioritize Quality Over Volume (Content Curation)**
The post-2015 bulk acquisition era lowered average IMDb scores. Recent years show a stabilization at lower volume but higher quality. Amazon should continue this quality-first strategy and actively de-prioritize low-rated content (IMDb < 5.0) in recommendation algorithms.

**3. Expand Regional Originals (Growth Markets)**
India is already the #2 content-producing country, proving that regional investment drives catalog growth and local subscriber acquisition. Amazon should replicate this model in Brazil, Southeast Asia, and Sub-Saharan Africa — high-growth, underserved streaming markets.

**4. Develop Family-Friendly Content (Subscriber Diversification)**
G and PG rated content is severely underrepresented. Adding quality family content would help Amazon Prime compete with Disney+ for the family subscriber segment — a loyal, long-tenure demographic.

**5. Leverage Top Talent for Exclusive Deals**
The most prolific actors and directors on the platform represent relationship capital. Exclusive multi-title deals with top-performing directors (high IMDb scores + high volume) would prevent talent migration to competitors and secure catalog differentiation.

**6. Build a Dual-Rating Recommendation Score**
IMDb and TMDb scores are correlated but not identical. A blended quality score (weighted combination of both, normalized by votes/popularity) would create a more reliable content ranking system — reducing the risk of burying good content that scores differently on each platform.

**7. Use Runtime Data for UI/UX Optimization**
Content under 30 minutes should be tagged as 'Quick Watch' in the UI. Very long movies (>150 min) should be surfaced with appropriate cues. Runtime-aware recommendations align viewer intent with content format, improving completion rates and satisfaction scores.

---

*These insights collectively enable Amazon Prime to improve subscriber retention, accelerate regional growth, optimize content investment, and strengthen its competitive position in the streaming market.*

---
